# Mundial 2026 — Predicciones en Vivo

**Secciones:** 1. Clasificaciones · 2. Backtest (partidos jugados) · 3. Próximos partidos

**Actualizar:** Re-ejecutar todas las celdas tras descargar datos nuevos.

In [ ]:
import sys, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore') 
 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from src.models.poisson_model import PoissonModel
from src.prediction.predict import predict_matches, format_predictions
from src.prediction.update import update_elo, save_elo, load_elo
from src.prediction.tracker import save_predictions, evaluate, summary_metrics
from src.simulation.tournament import GROUPS, compute_standings

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load models
with open('../models/poisson_params.json') as f:
    params = json.load(f)
elo_ratings = load_elo('../models/elo_ratings.json')
xgb = joblib.load('../models/xgb_pipeline.joblib')

poisson = PoissonModel()
poisson.params_ = params
poisson.teams_  = list(params['attack'].keys())

# Load raw match data
df_raw = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
wc26   = df_raw[(df_raw['tournament'] == 'FIFA World Cup') & (df_raw['date'].dt.year == 2026)].copy()
played  = wc26.dropna(subset=['home_score', 'away_score']).copy()
pending = wc26[wc26['home_score'].isna()].copy()

print(f'Partidos jugados  : {len(played)}')
print(f'Partidos pendientes: {len(pending)}')

Partidos jugados  : 44
Partidos pendientes: 28


## 1. Estado actual — Clasificaciones

In [2]:
print('Clasificaciones tras la Jornada 2 (partidos jugados)\n')
for group, teams in sorted(GROUPS.items()):
    gm = played[played['home_team'].isin(teams) & played['away_team'].isin(teams)]
    st = compute_standings(gm, teams)
    print(f'Grupo {group}')
    for rank, (_, r) in enumerate(st.iterrows(), 1):
        pj = int(r['w'] + r['d'] + r['l'])
        marker = ' <-- CLASIFICA (proyectado)' if rank <= 2 and pj == 2 else ''
        print(f'  {rank}. {r["team"]:26s}  {int(r["pts"]):2d}pts  {int(r["gf"]):2d}:{int(r["ga"]):2d}  ({pj}PJ){marker}')
    print()

Clasificaciones tras la Jornada 2 (partidos jugados)

Grupo A
  1. Argentina                    6pts   5: 0  (2PJ) <-- CLASIFICA (proyectado)
  2. Austria                      3pts   3: 3  (2PJ) <-- CLASIFICA (proyectado)
  3. Algeria                      3pts   2: 4  (2PJ)
  4. Jordan                       0pts   2: 5  (2PJ)

Grupo B
  1. United States                6pts   6: 1  (2PJ) <-- CLASIFICA (proyectado)
  2. Australia                    3pts   2: 2  (2PJ) <-- CLASIFICA (proyectado)
  3. Paraguay                     3pts   2: 4  (2PJ)
  4. Turkey                       0pts   0: 3  (2PJ)

Grupo C
  1. Egypt                        4pts   4: 2  (2PJ) <-- CLASIFICA (proyectado)
  2. Iran                         2pts   2: 2  (2PJ) <-- CLASIFICA (proyectado)
  3. Belgium                      2pts   1: 1  (2PJ)
  4. New Zealand                  1pts   3: 5  (2PJ)

Grupo D
  1. Canada                       4pts   7: 1  (2PJ) <-- CLASIFICA (proyectado)
  2. Switzerland                 

## 2. Backtest completo — todos los partidos jugados

In [3]:
md1_played = played[played['date'].dt.date <= pd.Timestamp('2026-06-17').date()]
md2_played = played[played['date'].dt.date >= pd.Timestamp('2026-06-18').date()]

def _backtest(df):
    p = predict_matches(list(zip(df['home_team'], df['away_team'])), poisson, xgb, elo_ratings, neutral=True)
    return summary_metrics(evaluate(p, df))

m1    = _backtest(md1_played)
m2    = _backtest(md2_played)
m_all = _backtest(played)

print(f"{'':24s} {'N':>4}  {'Resultado':>10}  {'Score exacto':>13}  {'MAE goles':>9}")
print('-' * 68)
for label, m in [('Jornada 1  (11-17 jun)', m1),
                 ('Jornada 2  (18-22 jun)', m2),
                 ('Total', m_all)]:
    n   = m['n_matches']
    ra  = m['result_accuracy']
    ep  = m['exact_score_pct']
    mae = (m['home_goal_mae'] + m['away_goal_mae']) / 2
    print(f"{label:24s} {n:>4d}  {ra:>8.1%} ({int(ra*n):2d}/{n})  {ep:>8.1%} ({int(ep*n):2d}/{n})  {mae:>8.2f}")

                            N   Resultado   Score exacto  MAE goles
--------------------------------------------------------------------
Jornada 1  (11-17 jun)     24     50.0% (12/24)      0.0% ( 0/24)      1.10
Jornada 2  (18-22 jun)     20     75.0% (15/20)     15.0% ( 3/20)      0.80
Total                      44     61.4% (27/44)      6.8% ( 3/44)      0.97


In [4]:
all_pairs = list(zip(played['home_team'], played['away_team']))
preds_all = predict_matches(all_pairs, poisson, xgb, elo_ratings, neutral=True)
ev_all    = evaluate(preds_all, played)

# Add date from played (evaluate() doesn't carry it through)
dates = played[['home_team', 'away_team', 'date']].copy()
ev_all = ev_all.merge(dates, on=['home_team', 'away_team'], how='left')

def _jornada(d):
    return 'J1' if d.date() <= pd.Timestamp('2026-06-17').date() else 'J2'

ev_all['J']     = ev_all['date'].apply(_jornada)
ev_all['Real']  = ev_all['home_score'].astype(int).astype(str) + '-' + ev_all['away_score'].astype(int).astype(str)
ev_all['Res']   = ev_all['result_correct'].map({True: 'OK', False: 'X'})
ev_all['Score'] = ev_all['exact_score_correct'].map({True: 'OK', False: 'X'})
ev_all['p_home_win'] = ev_all['p_home_win'].map('{:.0%}'.format)
ev_all['p_draw']     = ev_all['p_draw'].map('{:.0%}'.format)
ev_all['p_away_win'] = ev_all['p_away_win'].map('{:.0%}'.format)

out = ev_all[['J','home_team','away_team','Real','pred_score','Res','Score','p_home_win','p_draw','p_away_win']].copy()
out.columns = ['J','Local','Visitante','Real','Pred','Res','Sc','P(loc)','P(emp)','P(vis)']
pd.set_option('display.max_rows', 60)
print(out.to_string(index=False))

 J          Local              Visitante Real Pred Res Sc P(loc) P(emp) P(vis)
J1         Mexico           South Africa  2-0  1-0  OK  X    60%    27%    12%
J1    South Korea         Czech Republic  2-1  1-1  OK  X    42%    32%    26%
J1         Canada Bosnia and Herzegovina  1-1  2-0   X  X    77%    18%     5%
J1  United States               Paraguay  4-1  1-1   X  X    30%    29%    41%
J1          Qatar            Switzerland  1-1  0-2   X  X     8%    17%    75%
J1         Brazil                Morocco  1-1  0-0   X  X    39%    29%    32%
J1          Haiti               Scotland  0-1  1-1  OK  X    23%    28%    49%
J1      Australia                 Turkey  2-0  1-1  OK  X    44%    30%    26%
J1        Germany                Curaçao  7-1  3-0  OK  X    89%     8%     3%
J1    Ivory Coast                Ecuador  1-0  0-0   X  X    17%    36%    47%
J1    Netherlands                  Japan  2-2  1-1   X  X    27%    23%    50%
J1         Sweden                Tunisia  5-1  1-1  

## 3. Próximos partidos — Top 5 scores + probabilidades

In [5]:
from src.simulation.tournament import NAME_ALIASES

MAX_G = 5
all_pending = pending.sort_values('date').copy()
pending_pairs = list(zip(all_pending['home_team'], all_pending['away_team']))

# Ensemble outcome probs (single call for all matches)
preds_pending = predict_matches(pending_pairs, poisson, xgb, elo_ratings, neutral=True)

rows = []
for (home, away), (_, pred_row), date in zip(
    pending_pairs, preds_pending.iterrows(), all_pending['date'].values
):
    h_model = NAME_ALIASES.get(home, home)
    a_model = NAME_ALIASES.get(away, away)

    mat  = poisson.predict_score_matrix(h_model, a_model, neutral=True, max_goals=MAX_G)
    flat = mat.flatten()
    top_idxs = np.argsort(flat)[::-1][:5]

    entry = {
        'Fecha':     pd.Timestamp(date).strftime('%d %b'),
        'Local':     home,
        'Visitante': away,
    }
    for rank, idx in enumerate(top_idxs, 1):
        h_g, a_g = idx // (MAX_G + 1), idx % (MAX_G + 1)
        entry[f'#{rank}'] = f'{h_g}-{a_g} ({flat[idx]:.0%})'

    entry['P(loc)'] = f'{pred_row["p_home_win"]:.0%}'
    entry['P(emp)'] = f'{pred_row["p_draw"]:.0%}'
    entry['P(vis)'] = f'{pred_row["p_away_win"]:.0%}'
    rows.append(entry)

pd.set_option('display.max_colwidth', 14)
print(f'{len(rows)} partidos pendientes\n')
print(pd.DataFrame(rows).to_string(index=False))

# Save for future evaluation
save_predictions(preds_pending, phase='group_stage', matchday=0)

28 partidos pendientes

 Fecha                  Local      Visitante        #1        #2        #3        #4        #5 P(loc) P(emp) P(vis)
23 Jun               Portugal     Uzbekistan 1-0 (14%) 1-1 (13%) 2-0 (12%) 0-0 (12%)  2-1 (9%)    59%    25%    16%
23 Jun               Colombia       DR Congo 1-0 (19%) 0-0 (16%) 2-0 (14%) 1-1 (12%)  2-1 (8%)    63%    25%    12%
23 Jun                England          Ghana 1-0 (18%) 2-0 (17%) 0-0 (12%) 3-0 (11%) 1-1 (10%)    68%    21%    11%
23 Jun                 Panama        Croatia 1-1 (13%) 1-2 (10%)  0-2 (9%)  0-1 (9%)  0-0 (7%)    20%    25%    56%
24 Jun                 Mexico Czech Republic 1-0 (14%) 2-0 (14%) 1-1 (12%) 0-0 (10%)  2-1 (9%)    52%    27%    21%
24 Jun           South Africa    South Korea 0-1 (15%) 1-1 (13%) 0-0 (13%) 0-2 (12%)  1-2 (9%)    16%    28%    56%
24 Jun                 Canada    Switzerland 1-1 (15%) 0-0 (13%) 0-1 (11%) 1-0 (10%)  1-2 (7%)    30%    32%    38%
24 Jun Bosnia and Herzegovina          Qatar 1-1

WindowsPath('data/predictions/group_stage_md0.csv')

## Como usar

### Después de cada jornada (actualización ligera)
```python
# 1. Re-descargar datos
from src.data.download import download_all
download_all(force=True)

# 2. Actualizar ELO con los nuevos resultados
import pandas as pd
from src.prediction.update import update_elo, save_elo, load_elo
df = pd.read_csv('data/raw/results.csv', parse_dates=['date'])
nuevos = df[(df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2026)]
nuevos = nuevos.dropna(subset=['home_score', 'away_score'])  # solo jugados
elo = update_elo(nuevos, load_elo())
save_elo(elo)

# 3. Re-ejecutar el notebook completo
```

### Entre fases (fin de grupos → octavos, etc.)
```python
# Reentrenar Poisson + XGBoost con todos los datos disponibles (~2 min)
from src.prediction.update import retrain_full
poisson, xgb, elo_ratings = retrain_full()
```
Luego agregar los enfrentamientos de la siguiente fase en `pending_pairs` de la sección 3.

### Predecir un partido específico
```python
from src.prediction.predict import predict_matches, format_predictions
import json, joblib
from src.models.poisson_model import PoissonModel
from src.prediction.update import load_elo

with open('models/poisson_params.json') as f: params = json.load(f)
poisson = PoissonModel(); poisson.params_ = params; poisson.teams_ = list(params['attack'].keys())
xgb = joblib.load('models/xgb_pipeline.joblib')
elo = load_elo()

pred = predict_matches([('France', 'Argentina')], poisson, xgb, elo, neutral=True)
print(format_predictions(pred).to_string(index=False))
```

### Evaluar predicciones guardadas vs resultados reales
```python
from src.prediction.tracker import load_predictions, evaluate, summary_metrics
import pandas as pd

preds = load_predictions('group_stage', 0)       # cargar predicciones guardadas
results = pd.read_csv('data/raw/results.csv', parse_dates=['date'])
actuals = results[(results['tournament']=='FIFA World Cup') & results['home_score'].notna()]
ev = evaluate(preds, actuals)
print(summary_metrics(ev))
```